In [ ]:
import os
import sys
import logging

logging.basicConfig(stream=sys.stdout, level=logging.DEBUG)

os.environ["CUDA_VISIBLE_DEVICES"] = "7"
#os.environ["TRITON_DEBUG"] = "1"
#os.environ["FLASH_ATTENTION_TRITON_AMD_DEBUG"] = "1"
os.environ["FLASH_ATTENTION_TRITON_AMD_AUTOTUNE"] = os.environ["TRITON_PRINT_AUTOTUNING "] = "0"

import math
import time
from tqdm import tqdm
import torch

from transformers.modeling_flash_attention_utils import _flash_attention_forward, attention_vanilla_forward_pytorch_ref_impl
from transformers.triton_flash_attention_fp8_block import block_scaling_node, FIXED_BLOCK_M, FIXED_BLOCK_N
from transformers.triton_hadamard_transform import hadamard_transform

from torchtitan.tools.logging import logger

model_type = "deepseek-v2"
log_step = 10
max_step = 100
use_sdpa = False
exclude_input_cvt = True
use_fp8 = True
eval_bwd = True
with_outliers = True
use_hadamard = False
uniform_dist = False

# fa_fwd = torch.compile(
#     _flash_attention_forward,
#     fullgraph=True,
# )
fa_fwd = _flash_attention_forward

torch_dtype = torch.bfloat16
e4m3_dtype = torch.float8_e4m3fnuz
e5m2_dtype = torch.float8_e5m2fnuz
device = torch.device("cuda")

def rmse(x: torch.Tensor, y: torch.Tensor) -> torch.Tensor:
    return torch.sqrt(torch.mean((x - y)**2))

def mae(x: torch.Tensor, y: torch.Tensor) -> torch.Tensor:
    return torch.mean(torch.abs(x - y))

def prepare_data(model_type: str, device: torch.device, with_outliers: bool = False):
    torch.manual_seed(0)

    configs = {
        "opt-125m": {
            "seqlen": 2048,
            "n_head": 12,
            "n_head_kv": 12,
            "head_dim": 64,
        },
        "llama2-7b": {
            "seqlen": 4096,
            "n_head": 32,
            "n_head_kv": 32,
            "head_dim": 128,
        },
        "llama2-70b": {
            "seqlen": 4096,
            "n_head": 64,
            "n_head_kv": 64,
            "head_dim": 128,
        },
        "llama3-8b": {
            "seqlen": 8192,
            "n_head": 32,
            "n_head_kv": 8,
            "head_dim": 128,
        },
        "llama3-70b": {
            "seqlen": 8192,
            "n_head": 64,
            "n_head_kv": 8,
            "head_dim": 128,
        },
        "qwen3.5-mini": {
            "seqlen": 4096,
            "n_head": 32,
            "n_head_kv": 32,
            "head_dim": 96,
        },
        "deepseek-v2": {
            "seqlen": 4096,
            "n_head": 16,
            "n_head_kv": 16,
            "head_dim_qk": 192,
            "head_dim_v": 128,
        },
    }

    c=configs[model_type]
    seqlen = c["seqlen"]
    n_head = c["n_head"]
    n_head_kv = c["n_head_kv"]
    head_dim = c.get("head_dim", None)
    head_dim_qk = c.get("head_dim_qk", head_dim)
    head_dim_v = c.get("head_dim_v", head_dim)
    batch_size = 1
    query_states = torch.randn((batch_size, seqlen, n_head, head_dim_qk),
                               dtype=torch_dtype,
                               device=device)
    key_states = torch.randn((batch_size, seqlen, n_head_kv, head_dim_qk),
                             dtype=torch_dtype,
                             device=device)
    value_states = torch.randn((batch_size, seqlen, n_head_kv, head_dim_v),
                               dtype=torch_dtype,
                               device=device)
    sm_scale = 1 / math.sqrt(head_dim_qk)

    # outliers
    if with_outliers:
        if not uniform_dist:
            outlier_idx = torch.randint(0, head_dim_qk, (1, )).to(device=device)
            sequence_p = torch.ones((1, seqlen, 1, 1), device=device) * 0.001 #* batch_size * n_head * head_dim
            p_mask = torch.bernoulli(sequence_p)
            outlier_dist = 100 * torch.randn(size=(batch_size, seqlen, 1, 1)).to(
                dtype=torch_dtype, device=device)
            query_states[:, :, :, outlier_idx] += outlier_dist * p_mask
            outlier_dist = 100 * torch.randn(size=(batch_size, seqlen, 1, 1)).to(
                dtype=torch_dtype, device=device)
            key_states[:, :, :, outlier_idx] += outlier_dist * p_mask
            outlier_dist = 100 * torch.randn(size=(batch_size, seqlen, 1, 1)).to(
                dtype=torch_dtype, device=device)
            value_states[:, :, :, outlier_idx] += outlier_dist * p_mask
        else:
            r = 1 / (batch_size * n_head * head_dim_qk)

            p_mask = torch.bernoulli(
                torch.ones(batch_size,
                        seqlen,
                        n_head,
                        head_dim_qk,
                        dtype=torch_dtype,
                        device=device) * 0.001 * r)
            outlier_dist = 100 * torch.randn(size=(batch_size, seqlen, n_head,
                                                head_dim_qk)).to(dtype=torch_dtype,
                                                                device=device)
            query_states += outlier_dist * p_mask

            r = 1 / (batch_size * n_head_kv * head_dim_qk)
            p_mask = torch.bernoulli(
                torch.ones(batch_size,
                        seqlen,
                        n_head_kv,
                        head_dim_qk,
                        dtype=torch_dtype,
                        device=device) * 0.001 * r)
            outlier_dist = 100 * torch.randn(size=(batch_size, seqlen, n_head_kv,
                                                head_dim_qk)).to(dtype=torch_dtype,
                                                                device=device)
            key_states += outlier_dist * p_mask

            p_mask = torch.bernoulli(
                torch.ones(batch_size,
                        seqlen,
                        n_head_kv,
                        head_dim_v,
                        dtype=torch_dtype,
                        device=device) * 0.001* r)
            outlier_dist = 100 * torch.randn(size=(batch_size, seqlen, n_head_kv,
                                                head_dim_v)).to(dtype=torch_dtype,
                                                                device=device)
            value_states += outlier_dist * p_mask


    return query_states, key_states, value_states, sm_scale


def check_and_convert_fp8(t, descale):
    finfo = torch.finfo(e4m3_dtype)
    return ((t * descale).clamp(min=finfo.min, max=finfo.max).to(e4m3_dtype)
            if t.dtype != e4m3_dtype else t)

query_states, key_states, value_states, sm_scale = prepare_data(model_type, device, with_outliers)
query_states_hp = query_states.clone().detach()
key_states_hp = key_states.clone().detach()
value_states_hp = value_states.clone().detach()

if eval_bwd:
    query_states = torch.nn.Parameter(query_states, requires_grad=True)
    key_states = torch.nn.Parameter(key_states, requires_grad=True)
    value_states = torch.nn.Parameter(value_states, requires_grad=True)
    query_states_hp = torch.nn.Parameter(query_states_hp, requires_grad=True)
    key_states_hp = torch.nn.Parameter(key_states_hp, requires_grad=True)
    value_states_hp = torch.nn.Parameter(value_states_hp, requires_grad=True)

#def step(query_states, key_states, value_states, query_states_hp, key_states_hp, value_states_hp, do_eval=False):
do_eval = True
if use_hadamard:
    query_states_r = hadamard_transform(query_states)
    key_states_r = hadamard_transform(key_states)
else:
    query_states_r = query_states
    key_states_r = key_states

with torch.no_grad():
    # range_q = torch.max(torch.abs(query_states_r))
    # range_k = torch.max(torch.abs(key_states_r))
    range_v = torch.max(torch.abs(value_states))

    query_length = query_states.shape[1]
    descale_q = descale_k = descale_v = None
    if use_fp8:
        dtype_max = torch.finfo(e4m3_dtype).max
        #descale_q = dtype_max / range_q
        #descale_k = dtype_max / range_k
        descale_v = dtype_max / range_v
        if exclude_input_cvt:
            # query_states_r_fp8 = check_and_convert_fp8(query_states_r, descale_q)
            # key_states_r_fp8 = check_and_convert_fp8(key_states_r, descale_k)
            query_states_r_fp8, descale_q = block_scaling_node(
                query_states_r, FIXED_BLOCK_M)
            key_states_r_fp8, descale_k = block_scaling_node(
                key_states_r, FIXED_BLOCK_N)
            value_states_fp8 = check_and_convert_fp8(value_states, descale_v)
    else:
        query_states_r_fp8 = query_states_r
        key_states_r_fp8 = key_states_r
        value_states_fp8 = value_states

    #print(f"descale_q={descale_q}, descale_k={descale_k}, descale_v={descale_v}")

if not use_sdpa:
    fa_output = fa_fwd(
        query_states_r,
        key_states_r,
        value_states,
        query_states_r_fp8,
        key_states_r_fp8,
        value_states_fp8,
        None,
        query_length,
        True,
        0.0,
        softmax_scale=sm_scale,
        descale_q=descale_q,
        descale_k=descale_k,
        descale_v=descale_v,
    )
else:
    fa_output = torch.nn.functional.scaled_dot_product_attention(
        query_states_r.transpose(1, 2),
        key_states_r.transpose(1, 2),
        value_states.transpose(1, 2),
        dropout_p=0.0,
        is_causal=True,
        scale=sm_scale,
        enable_gqa=True,
    ).transpose(1,2)

if do_eval:
    print(fa_output.dtype)
    fa_output_hp = attention_vanilla_forward_pytorch_ref_impl(
        query_states_hp, key_states_hp, value_states_hp, sm_scale, True, "bshd",
        False)[0]

if eval_bwd:
    loss = fa_output.mean()  #loss_fn(fa_output, output_states)
    loss.backward()

    if do_eval:
        loss_hp = fa_output_hp.mean() #loss_fn(fa_output_hp, output_states)
        loss_hp.backward()

if do_eval:
    e = rmse(fa_output, fa_output_hp)
    ae = mae(fa_output, fa_output_hp)
    snr = 10 * torch.log10(
        torch.mean(fa_output_hp**2) / torch.mean((fa_output - fa_output_hp)**2))
    print(
        f"fwd rmse(o)={e}, mae={ae}, relative={ae / torch.mean(fa_output_hp)}, snr={snr}"
    )
    if eval_bwd:
        e = rmse(query_states.grad, query_states_hp.grad)
        ae = mae(query_states.grad, query_states_hp.grad)
        snr = 10 * torch.log10(
            torch.mean(query_states_hp.grad**2) / torch.mean(
                (query_states.grad - query_states_hp.grad)**2))
        print(
            f"bwd rmse(dq)={e}, mae={ae}, relative={ae / torch.mean(query_states_hp)}, snr={snr}"
        )
        e = rmse(key_states.grad, key_states_hp.grad)
        ae = mae(key_states.grad, key_states_hp.grad)
        snr = 10 * torch.log10(
            torch.mean(key_states_hp.grad**2) / torch.mean(
                (key_states.grad - key_states_hp.grad)**2))
        print(
            f"bwd rmse(dk)={e}, mae={ae}, relative={ae / torch.mean(key_states_hp)}, snr={snr}"
        )
        e = rmse(value_states.grad, value_states_hp.grad)
        ae = mae(value_states.grad, value_states_hp.grad)
        snr = 10 * torch.log10(
            torch.mean(value_states_hp.grad**2) / torch.mean(
                (value_states.grad - value_states_hp.grad)**2))
        print(
            f"bwd rmse(dv)={e}, mae={ae}, relative={ae / torch.mean(value_states_hp)}, snr={snr}"
        )

#step(query_states, key_states, value_states, query_states_hp, key_states_hp, value_states_hp, do_eval=True)

In [ ]:
# torch.cuda.synchronize()
# time_last_log = time.perf_counter()
# for step_idx in tqdm(range(max_step)):
#     step(query_states, key_states, value_states, query_states_hp, key_states_hp, value_states_hp)
# torch.cuda.synchronize()
# time_now = time.perf_counter()
# print(f"average time/step={((time_now - time_last_log) / max_step) * 1000:.2f}ms")

In [ ]:
torch.cuda.synchronize()
time_last_log = time.perf_counter()
for i in tqdm(range(max_step)):
    if not use_sdpa:
        fa_output = fa_fwd(
            query_states_r,
            key_states_r,
            value_states,
            query_states_r_fp8,
            key_states_r_fp8,
            value_states_fp8,
            None,
            query_length,
            True,
            0.0,
            softmax_scale=sm_scale,
            descale_q=descale_q,
            descale_k=descale_k,
            descale_v=descale_v,
        )
    else:
        fa_output = torch.nn.functional.scaled_dot_product_attention(
            query_states_r.transpose(1, 2),
            key_states_r.transpose(1, 2),
            value_states.transpose(1, 2),
            dropout_p=0.0,
            is_causal=True,
            scale=sm_scale,
            enable_gqa=True,
        ).transpose(1, 2)
    if eval_bwd:
        loss = fa_output.mean()  #loss_fn(fa_output, output_states)
        loss.backward()

torch.cuda.synchronize()
time_now = time.perf_counter()
print(
    f"average time/step={((time_now - time_last_log) / max_step) * 1000:.2f}ms"
)


In [ ]:
from transformers.triton_flash_attention_fp8 import attn_fwd, _bwd_kernel

cached_kernel = (list(_bwd_kernel.fn.cache.values())[0])
kernel = (list(cached_kernel.values())[0])
# kernel = list(attn_fwd.fn.device_caches[0][0].values())[0]
# #print(print(_bwd_kernel.fn.device_caches[0]))
# # print(print(
# #     list(_bwd_kernel.fn.device_caches[0][0].values())[0].asm['amdgcn']))
# # # _bwd_kernel.fn.compile()
# print(kernel.asm.keys())

In [ ]:
print(kernel.asm['amdgcn'])